In [1]:
from pyspark.sql.functions import (
    col, trim, upper, when, current_timestamp,
    lit, to_date, sha2
)
from pyspark.sql.types import DecimalType, IntegerType
from datetime import datetime

BRONZE_TABLE  = "Interac_Bronze.dbo.merchants"
SILVER_TABLE  = "silver_merchants"
SILVER_DB     = "Interac_Fabric_Workspace.Interac_Silver.dbo"
PIPELINE_NAME = "NB_03_Silver_Merchants"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Silver Merchants Pipeline")
print(f"Started: {datetime.now()}")

StatementMeta(, 1ebb2d68-4f42-499f-8db2-6cd37747729e, 3, Finished, Available, Finished, False)

Silver Merchants Pipeline
Started: 2026-05-06 00:19:21.436121


In [2]:
df_bronze = spark.read.table(BRONZE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze rows read: {total_bronze:,}")

dq_results = {}
dq_results["null_merchant_id"] = df_bronze.filter(col("merchant_id").isNull()).count()
dq_results["null_merchant_name"] = df_bronze.filter(col("merchant_name").isNull()).count()
dq_results["null_province"] = df_bronze.filter(col("province").isNull()).count()
dq_results["suspended_merchants"] = df_bronze.filter(col("status") == "SUSPENDED").count()
dq_results["terminated_merchants"] = df_bronze.filter(col("status") == "TERMINATED").count()
dq_results["high_risk_merchants"] = df_bronze.filter(col("risk_rating") == "HIGH").count()
dq_results["interac_disabled"] = df_bronze.filter(col("interac_debit_enabled") == "N").count()

print("\nDQ CHECK RESULTS:")
print("-" * 45)
for check, count_val in dq_results.items():
    status = "⚠ FLAGGED" if count_val > 0 else "✓ PASSED"
    print(f"{check:<35} {count_val:>6,}  {status}")

StatementMeta(, 1ebb2d68-4f42-499f-8db2-6cd37747729e, 4, Finished, Available, Finished, False)

Bronze rows read: 2,000

DQ CHECK RESULTS:
---------------------------------------------
null_merchant_id                         0  ✓ PASSED
null_merchant_name                       0  ✓ PASSED
null_province                            0  ✓ PASSED
suspended_merchants                     85  ⚠ FLAGGED
terminated_merchants                    34  ⚠ FLAGGED
high_risk_merchants                    109  ⚠ FLAGGED
interac_disabled                        57  ⚠ FLAGGED


In [3]:
df_quarantine = df_bronze.filter(
    col("merchant_id").isNull() | col("merchant_name").isNull()
)
quarantine_count = df_quarantine.count()

if quarantine_count > 0:
    (df_quarantine
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY_OR_NAME"))
        .withColumn("_quarantined_at", current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER_DB}.silver_merchants_quarantine"))
    print(f"Quarantined: {quarantine_count:,} records")

df_valid = df_bronze.filter(
    col("merchant_id").isNotNull() & col("merchant_name").isNotNull()
)

df_silver = (df_valid
    .withColumn("merchant_id",    trim(col("merchant_id")))
    .withColumn("merchant_name",  trim(col("merchant_name")))
    .withColumn("category_code",  upper(trim(col("category_code"))))
    .withColumn("category_name",  trim(col("category_name")))
    .withColumn("city",           trim(col("city")))
    .withColumn("province",       upper(trim(col("province"))))
    .withColumn("acquiring_bank", trim(col("acquiring_bank")))
    .withColumn("risk_rating",    upper(trim(col("risk_rating"))))
    .withColumn("status",         upper(trim(col("status"))))
    .withColumn("account_number_masked",
        sha2(col("account_number").cast("string"), 256))
    .withColumn("transit_number_masked",
        sha2(col("transit_number").cast("string"), 256))
    .withColumn("institution_number_masked",
        sha2(col("institution_number").cast("string"), 256))
    .drop("account_number", "transit_number", "institution_number")
    .withColumn("registration_date",
        to_date(col("registration_date"), "yyyy-MM-dd"))
    .withColumn("last_modified_date",
        to_date(col("last_modified_date"), "yyyy-MM-dd"))
    .withColumn("avg_transaction_amount",
        col("avg_transaction_amount").cast(DecimalType(18, 2)))
    .withColumn("monthly_volume_estimate",
        col("monthly_volume_estimate").cast(IntegerType()))
    .withColumn("is_active",
        when(col("status") == "ACTIVE", "Y").otherwise("N"))
    .withColumn("is_high_risk",
        when(col("risk_rating") == "HIGH", "Y").otherwise("N"))
    .withColumn("_silver_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",    lit(PIPELINE_NAME))
    .withColumn("_batch_date",       lit(BATCH_DATE))
    .withColumn("_is_current",       lit("Y"))
    .drop("_ingested_at", "_source_file", "_lakehouse")
)

print(f"Valid records: {df_silver.count():,}")

StatementMeta(, 1ebb2d68-4f42-499f-8db2-6cd37747729e, 5, Finished, Available, Finished, False)

Valid records: 2,000


In [4]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}"))

spark.sql(f"OPTIMIZE {SILVER_DB}.{SILVER_TABLE} ZORDER BY (merchant_id, province)")

final_count = spark.read.table(f"{SILVER_DB}.{SILVER_TABLE}").count()

print("\n" + "="*60)
print("SILVER MERCHANTS SUMMARY")
print("="*60)
print(f"Bronze rows in    : {total_bronze:,}")
print(f"Quarantined       : {quarantine_count:,}")
print(f"Silver rows out   : {final_count:,}")
print(f"Pass rate         : {round(final_count/total_bronze*100, 2)}%")
print(f"PII masked        : account_number, transit_number, institution_number")
print(f"Table             : {SILVER_DB}.{SILVER_TABLE}")
print(f"Completed at      : {datetime.now()}")
print("="*60)

StatementMeta(, 1ebb2d68-4f42-499f-8db2-6cd37747729e, 6, Finished, Available, Finished, False)


SILVER MERCHANTS SUMMARY
Bronze rows in    : 2,000
Quarantined       : 0
Silver rows out   : 2,000
Pass rate         : 100.0%
PII masked        : account_number, transit_number, institution_number
Table             : Interac_Fabric_Workspace.Interac_Silver.dbo.silver_merchants
Completed at      : 2026-05-06 00:20:12.393547
